# Smart Keyboard: train and evaluate on Google Colab

Runs the whole pipeline of the repository on a free GPU. Before you start:
**Runtime → Change runtime type → T4 GPU**.

A Colab machine is deleted when the session ends, so the weights are saved in two ways: on Google Drive
(if `USE_DRIVE = True`) and on the Hugging Face Hub (step 6, right after training). After a reset, run the
setup cells, `prepare`, and the download cell; nothing has to be trained again.

## 1. Setup

In [ ]:
USE_DRIVE = True     # False: keep the files on this Colab machine only (lost when the session ends)
REPO_URL = 'https://github.com/Enes-Ozbek/smart-keyboard.git'
HF_REPO = 'Waxech/smart-keyboard-models'    # Hugging Face model repo for the weights

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE = '/content/drive/MyDrive/smart-keyboard'
else:
    SAVE = '/content/work'

In [ ]:
import os
if not os.path.exists('/content/smart-keyboard'):
    !git clone $REPO_URL /content/smart-keyboard
%cd /content/smart-keyboard
!git pull
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# every command below uses the same folders
ARGS = f'--artifacts-dir {SAVE}/artifacts --results-dir {SAVE}/results'

## 2. Data and the classical baseline (area B)

`prepare` also has to run again after a reset (the data folder is not saved).

In [ ]:
!python -m keyboard.cli prepare $ARGS

In [ ]:
!python -m keyboard.cli bpe $ARGS

In [ ]:
!python -m keyboard.cli ngram $ARGS

## 3. Feedforward neural language model (area A)

In [ ]:
!python -m keyboard.cli ffnn $ARGS

## 4. Fine-tune GPT-2 small (area C)

In [ ]:
!python -m keyboard.cli gpt2 $ARGS

## 5. INT8 quantization (area D)

Prints a small study on validation text: FP32, plain INT8, per-channel INT8, SmoothQuant, and (if needed)
how many layers stay in FP32. The numbers are also saved in `artifacts/gpt2-int8/quantization.json`.

In [ ]:
!python -m keyboard.cli quantize $ARGS

## 6. Save the weights to the Hugging Face Hub (R7: no weights in git)

Needs a *write* token from https://huggingface.co/settings/tokens. Run this as soon as training is done.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
!python -m keyboard.cli upload $ARGS --models-repo $HF_REPO

Only after a reset: get the weights back instead of training again (run setup and `prepare` first).

In [ ]:
!python -m keyboard.cli download $ARGS --models-repo $HF_REPO

## 7. Evaluate all models on the test split

Perplexity of the FP32 models uses the GPU; top-3 accuracy and latency run on the CPU of this Colab machine
(written under the table). Every model makes 1,000 timed predictions, so this takes about 40 minutes. Each line
saves its results when it finishes; run all of them in the same session so the latencies come from the same CPU.

In [ ]:
!python -m keyboard.cli evaluate $ARGS --models ngram-addone ngram-kn ffnn
!python -m keyboard.cli evaluate $ARGS --models gpt2-fp32
!python -m keyboard.cli evaluate $ARGS --models gpt2-int8
print(open(f'{SAVE}/results/results.md').read())

In [ ]:
from google.colab import files     # results.md goes into the results/ folder of the GitHub repo
files.download(f'{SAVE}/results/results.md')
files.download(f'{SAVE}/artifacts/gpt2-int8/quantization.json')

## 8. Try the app (public link)

In [ ]:
!python app.py --share --artifacts-dir $SAVE/artifacts